# LTSR Colab Phase 0 — setup / preflight

Python 3.10、CUDA、依存関係、checkpoint、DREAM4、GSE112372を準備する。

実行順は、Python 3.10確認 → Drive mount/source固定 → Phaseセルである。
Python環境導入でruntimeが再起動した場合は、再接続して最初のセルからやり直す。

In [ ]:
# 必ず最初に実行する。Colab UI kernelとは別に研究コード用Python 3.10を用意する。
import hashlib
import subprocess
import sys
import urllib.request
from pathlib import Path

PY310_ROOT = Path("/content/ltsr-py310")
PY310 = PY310_ROOT / "bin" / "python"
INSTALLER = Path("/content/Miniconda3-py310_23.11.0-2-Linux-x86_64.sh")
INSTALLER_URL = (
    "https://repo.anaconda.com/miniconda/"
    "Miniconda3-py310_23.11.0-2-Linux-x86_64.sh"
)
INSTALLER_SHA256 = (
    "35a58b8961e1187e7311b979968662c6223e86e1451191bed2e67a72b6bd0658"
)

def worker_version():
    if not PY310.is_file():
        return None
    return subprocess.check_output(
        [
            str(PY310), "-c",
            "import sys; print('.'.join(map(str, sys.version_info[:3])))",
        ],
        text=True,
    ).strip()

version = worker_version()
print("Colab controller:", sys.version)
print("LTSR worker before setup:", version)
if version is None or not version.startswith("3.10."):
    if not INSTALLER.is_file():
        urllib.request.urlretrieve(INSTALLER_URL, INSTALLER)
    digest = hashlib.sha256(INSTALLER.read_bytes()).hexdigest()
    if digest != INSTALLER_SHA256:
        raise RuntimeError(
            f"Miniconda installer checksum mismatch: {digest}"
        )
    subprocess.run(
        [
            "bash", str(INSTALLER), "-b", "-u",
            "-p", str(PY310_ROOT),
        ],
        check=True,
    )
    version = worker_version()
if version is None or not version.startswith("3.10."):
    raise RuntimeError(f"Python 3.10 worker setup failed: {version}")
print("LTSR worker Python 3.10: OK —", version)

In [ ]:
# Google認証とDrive mountはユーザー自身が行う。
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LTSR_colab")
REPO_ROOT = Path("/content/LTSR")
BRANCH = "20260726/gpu-scale-prep-colab"
REPO_URL = (
    "https://github.com/blabo25226/"
    "Layer-selective_Transformer-based_Symbolic_Regression.git"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
lock_path = DRIVE_ROOT / "source_lock.json"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if lock_path.is_file():
    locked_commit = json.loads(lock_path.read_text(encoding="utf-8"))["commit"]
    subprocess.run(["git", "checkout", "--detach", locked_commit], cwd=REPO_ROOT, check=True)
else:
    locked_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)

sys.path.insert(0, str(REPO_ROOT / "src"))
from colab_runtime import assert_locked_source, require_python_310

require_python_310(PY310)
print("locked commit:", assert_locked_source(REPO_ROOT, DRIVE_ROOT))
print("Drive root:", DRIVE_ROOT)

## 依存関係

CUDA版PyTorchを先に固定し、その後GPU requirementsとNeSymReSを導入する。
`requirements/dev.txt`はCPU版torchで上書きするため使用しない。

In [ ]:
import subprocess
import sys

commands = [
    [str(PY310), "-m", "pip", "install", "--upgrade", "pip"],
    [
        str(PY310), "-m", "pip", "install", "torch==2.5.1",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ],
    [str(PY310), "-m", "pip", "install", "-r", "requirements/gpu.txt"],
    [str(PY310), "-m", "pip", "install", "-e", "NSRS/src"],
    [str(PY310), "-m", "pip", "install", "pytest", "pysr"],
]
for command in commands:
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, check=True)

## checkpointと外部データをDriveへcache

In [ ]:
import shutil
import tarfile
import urllib.request

checkpoint = DRIVE_ROOT / "checkpoints" / "100M.ckpt"
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.is_file():
    urllib.request.urlretrieve(
        "https://huggingface.co/TommasoBendinelli/"
        "NeuralSymbolicRegressionThatScales/resolve/main/100M.ckpt",
        checkpoint,
    )
print("checkpoint bytes:", checkpoint.stat().st_size)

dream4_archive = DRIVE_ROOT / "data" / "dream4.tar.gz"
if not dream4_archive.is_file():
    work = Path("/content/dream4_setup")
    work.mkdir(parents=True, exist_ok=True)
    zip_path = work / "dream4.zip"
    urllib.request.urlretrieve(
        "https://gnw.sourceforge.net/resources/"
        "DREAM4%20in%20silico%20challenge.zip",
        zip_path,
    )
    shutil.unpack_archive(zip_path, work / "extracted")
    size10 = next((work / "extracted").rglob("Size 10"))
    data_root = REPO_ROOT / "data" / "dream4"
    data_root.mkdir(parents=True, exist_ok=True)
    shutil.copytree(size10.parent, data_root, dirs_exist_ok=True)
    dream4_archive.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(dream4_archive, "w:gz") as handle:
        handle.add(data_root, arcname="dream4")

# Phase 8 download is executed before manifest creation so fingerprints exist.
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
subprocess.run(
    [
        str(PY310), "-c",
        "from pathlib import Path; "
        "from data.human import prepare_gse112372; "
        "p=prepare_gse112372(Path('data/human/gse112372_lps')); "
        "print(p.source, sorted(p.X_donors))",
    ],
    cwd=REPO_ROOT,
    env=env,
    check=True,
)
human_archive = DRIVE_ROOT / "data" / "gse112372_lps.tar.gz"
human_archive.parent.mkdir(parents=True, exist_ok=True)
with tarfile.open(human_archive, "w:gz") as handle:
    handle.add(
        REPO_ROOT / "data" / "human" / "gse112372_lps",
        arcname="gse112372_lps",
    )
print("cached:", checkpoint, dream4_archive, human_archive)

## GPU preflightとテスト

In [ ]:
from colab_runtime import restore_static_assets, run_command

restore_static_assets(REPO_ROOT, DRIVE_ROOT)
run_command(
    REPO_ROOT,
    [
        str(PY310), "scripts/preflight_gpu.py",
        "--weights", "NSRS/weights/100M.ckpt",
        "--config", "NSRS/jupyter/100M/config.yaml",
        "--eq-setting", "NSRS/jupyter/100M/eq_setting.json",
    ],
)
run_command(
    REPO_ROOT,
    [str(PY310), "-m", "compileall", "-q", "src", "scripts", "tests"],
)
run_command(REPO_ROOT, ["bash", "-n", "scripts/run_gpu_pipeline.sh"])
run_command(REPO_ROOT, [str(PY310), "-m", "pytest", "-q"])